In [1]:
import pandas as pd
import re
from pathlib import Path

filename = Path("experiments") / "linear-california"
n_experiments = 100

max_d_T = 0
max_d_E = 0


def parse_value(v):
    # handles floats and "tensor(...)" strings
    if isinstance(v, (float, int)):
        return float(v)

    v = str(v)
    match = re.search(r"tensor\(([-0-9.eE]+)", v)
    if match:
        return float(match.group(1))

    return float(v)


for i in range(n_experiments):
    file_path = filename / f"model_{i}.csv"

    try:
        df = pd.read_csv(file_path)

        # parse values
        df["value"] = df["value"].apply(parse_value)

        # index for fast lookup
        df = df.set_index("constant type")

        tl1 = df.loc["trivial_l1", "value"]
        tl2 = df.loc["trivial_l2", "value"]
        duT = df.loc["dmoc_union_T", "value"]
        duE = df.loc["dmoc_union_E", "value"]

        diff_T = abs(duT - tl1)
        diff_E = abs(duE - tl2)
        max_d_T = max(max_d_T, diff_T)
        max_d_E = max(max_d_E, diff_E)

    except Exception as e:
        print(f"Skipping model_{i}.csv due to error: {e}")

print("Max |dmoc_union_T - trivial_l1|:", max_d_T)
print("Max |dmoc_union_E - trivial_l2|:", max_d_E)


Max |dmoc_union_T - trivial_l1|: 0.03912147168534996
Max |dmoc_union_E - trivial_l2|: 0.011203391394116968


In [ ]:
import numpy as np 
from pathlib import Path
import sys
sys.path.append(str(Path("fmca") / "build" / "py"))
import FMCA
import matplotlib.pyplot as plt 

base_path = Path("data")
filename = base_path / "MNIST"

#load X train, union (ignore labels)
def pairwise_distances(P):
    # P shape: (d, n)
    diff = P[:, :, None] - P[:, None, :]
    D = np.sqrt((diff**2).sum(axis=0))
    return D



X_train = np.loadtxt(filename/("X_train.csv"), delimiter=",",ndmin=2)
X_test = np.loadtxt(filename/("X_test.csv"), delimiter=",",ndmin=2)
Y_train = np.loadtxt(filename/("Y_train.csv"), delimiter="," , ndmin=2)
Y_test = np.loadtxt(filename/("Y_test.csv"), delimiter="," , ndmin=2)

X_union = np.vstack([X_train, X_test])
Y_union = np.vstack([Y_train, Y_test])
norm = "EUCLIDEAN"
#compute min pairwise distance (of distinct elements)
for X,Y in [(X_train, Y_train)]:
    X= X.transpose()
    Y=Y.transpose()

    dmoc = FMCA.DiscreteModulusOfContinuity()
    dmoc.init(X,Y, 1.64,51.55, 1000, norm, norm)
    data_m = dmoc.omegat()
    t_values = dmoc.tgrid()
    tx = dmoc.TX()
    qx = dmoc.qX()
    assert(t_values[0] == qx)
    assert(t_values[-1] == tx)

    idx = np.where(np.array(data_m) > 0)[0][0]
    print(idx)
    t = t_values[idx]
    print(t)

    for j in range(X.shape[1]):
        for l in range(j+1, X.shape[0]):
            x= X[:, j]
            y= X[:, l]
            d = np.linalg.norm(x - y)
            if (np.abs(d-t)< 1e-4):
                img1 = x.reshape(28, 28)
                img2 = x.reshape(28, 28)
                fig, axes = plt.subplots(1, 2)
        
                axes[0].imshow(img1, cmap="gray")
                axes[0].set_title(f"l={l}")
                axes[0].axis("off")

                axes[1].imshow(img2, cmap="gray")
                axes[1].set_title(f"j={j}")
                axes[1].axis("off")

                plt.suptitle(f"Distance ≈ {t:.4f}")
                plt.show()


        


print("end")

0
51.55
end


In [ ]:
import numpy as np 
from pathlib import Path
import sys
sys.path.append(str(Path("fmca") / "build" / "py"))
import FMCA
import matplotlib.pyplot as plt 

base_path = Path("data")
filename = base_path / "iris"

#load X train, union (ignore labels)
def pairwise_distances(P):
    # P shape: (d, n)
    diff = P[:, :, None] - P[:, None, :]
    D = np.sqrt((diff**2).sum(axis=0))
    return D



X_train = np.loadtxt(filename/("X_train.csv"), delimiter=",",ndmin=2)
X_test = np.loadtxt(filename/("X_test.csv"), delimiter=",",ndmin=2)
Y_train = np.loadtxt(filename/("Y_train.csv"), delimiter="," , ndmin=2)
F_train = np.loadtxt(filename/("F_train_3.csv"), delimiter="," , ndmin=2)

norm = "EUCLIDEAN"
#compute min pairwise distance (of distinct elements)
for X,Y, F in [(X_train, Y_train, F_train)]:
    X= X.transpose()
    Y=Y.transpose()
    F=F.transpose()

    unique_cols = np.unique(Y, axis=1)
    print(unique_cols)

    unique_cols = np.unique(F, axis=1)
    print(unique_cols)
    exit()
    


        


print("end")

[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
[[0.33404762]
 [0.33146793]
 [0.33448452]]
end


: 